In [1]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/capstone-ml'
DATA = f'{ROOT}/data'
PRED = f'{ROOT}/predictions'
MODELS = f'{ROOT}/models'
LOGS = f'{ROOT}/logs'

import sys, importlib.metadata as md, torch
print('Python', sys.version.split()[0])
for p in ['torch', 'transformers', 'accelerate', 'datasets', 'scikit-learn', 'pandas', 'numpy']:
    print(f'{p}=={md.version(p)}')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, switch the runtime to T4')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python 3.13.16
torch==2.14.1+cu130
transformers==5.22.1
accelerate==1.15.0
datasets==4.8.5
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.4.6
GPU: Tesla T4


In [2]:
import json, os
import pandas as pd

splits = {s: pd.read_csv(f'{DATA}/sst5_{s}.csv') for s in ['train', 'val', 'test']}
assert splits['train']['label'].value_counts().sort_index().tolist() == [3310, 1624, 3610]

with open(f'{DATA}/class_weights.json') as f:
    CLASS_WEIGHTS = {int(k): v for k, v in json.load(f)['weights'].items()}
print({s: len(df) for s, df in splits.items()}, CLASS_WEIGHTS)

{'train': 8544, 'val': 1101, 'test': 2210} {0: 0.8604, 1: 1.7537, 2: 0.7889}


In [3]:
from transformers import AutoTokenizer

CKPT = 'roberta-base'
MODEL_NAME = 'roberta'
tok = AutoTokenizer.from_pretrained(CKPT)

examples = ["it doesn't quite work, but the fundi was punctual and courteous.",
            "overcharged for the rewiring and left without explaining the invoice.",
            splits['train']['text'][2]]
for t in examples:
    print(t)
    print('  ->', tok.tokenize(t), '\n')

ids = tok(splits['train']['text'].tolist())['input_ids']
print(f'Longest training sentence: {max(len(i) for i in ids)} tokens')
print(f'Unknown tokens in all of training: {sum(i.count(tok.unk_token_id) for i in ids)}')

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

it doesn't quite work, but the fundi was punctual and courteous.
  -> ['it', 'Ġdoesn', "'t", 'Ġquite', 'Ġwork', ',', 'Ġbut', 'Ġthe', 'Ġfund', 'i', 'Ġwas', 'Ġpunct', 'ual', 'Ġand', 'Ġcour', 'te', 'ous', '.'] 

overcharged for the rewiring and left without explaining the invoice.
  -> ['over', 'charged', 'Ġfor', 'Ġthe', 'Ġre', 'w', 'iring', 'Ġand', 'Ġleft', 'Ġwithout', 'Ġexplaining', 'Ġthe', 'Ġinvoice', '.'] 

they presume their audience won't sit still for a sociology lesson, however entertainingly presented, so they trot out the conventional science-fiction elements of bug-eyed monsters and futuristic women in skimpy clothes.
  -> ['they', 'Ġpresume', 'Ġtheir', 'Ġaudience', 'Ġwon', "'t", 'Ġsit', 'Ġstill', 'Ġfor', 'Ġa', 'Ġsociology', 'Ġlesson', ',', 'Ġhowever', 'Ġentertaining', 'ly', 'Ġpresented', ',', 'Ġso', 'Ġthey', 'Ġtro', 't', 'Ġout', 'Ġthe', 'Ġconventional', 'Ġscience', '-', 'fiction', 'Ġelements', 'Ġof', 'Ġbug', '-', 'eyed', 'Ġmonsters', 'Ġand', 'Ġfuturistic', 'Ġwomen', 'Ġin', 'Ġs

In [4]:
from datasets import Dataset

MAX_LEN = 128

def to_hf(df):
    ds = Dataset.from_pandas(df[['id', 'text', 'label']])
    return ds.map(lambda batch: tok(batch['text'], truncation=True, max_length=MAX_LEN), batched=True)

hf = {s: to_hf(df) for s, df in splits.items()}
print(hf['train'])

Map:   0%|          | 0/8544 [00:00<?, ? examples/s]

Map:   0%|          | 0/1101 [00:00<?, ? examples/s]

Map:   0%|          | 0/2210 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 8544
})


## Stability rule (fixed before any seed was trained, 10 November 2026)

All three seeds train at learning rate 2e-5. A seed counts as collapsed if either holds:
- its validation macro-F1 never exceeds 0.40 in any of the four epochs, or
- its epoch-1 training loss is above 1.00.

If any seed collapses, all three seeds are retrained at 1e-5, and both runs are reported.
Otherwise the 2e-5 results stand as they are, whatever they show.

In [5]:
import math, time, shutil
import numpy as np
import torch
import torch.nn.functional as F
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, set_seed)
from sklearn.metrics import f1_score, accuracy_score

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
WEIGHTS = torch.tensor([CLASS_WEIGHTS[i] for i in range(3)], dtype=torch.float)
BATCH, EPOCHS, LR = 16, 4, 2e-5
STEPS_PER_EPOCH = math.ceil(len(hf['train']) / BATCH)
WARMUP = int(0.1 * STEPS_PER_EPOCH * EPOCHS)
print(f'{STEPS_PER_EPOCH} steps per epoch, {STEPS_PER_EPOCH * EPOCHS} in total, {WARMUP} warm-up steps, learning rate {LR}')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs['labels']
        outputs = model(**{k: v for k, v in inputs.items() if k != 'labels'})
        loss = F.cross_entropy(outputs.logits.float(), labels, weight=WEIGHTS.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro'), 'accuracy': accuracy_score(labels, preds)}

collator = DataCollatorWithPadding(tok)

def make_trainer(seed, out_dir, lr=LR, **overrides):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        CKPT, num_labels=3, id2label=ID2LABEL, label2id={v: k for k, v in ID2LABEL.items()})
    settings = dict(output_dir=out_dir, learning_rate=lr, per_device_train_batch_size=BATCH,
                    per_device_eval_batch_size=64, num_train_epochs=EPOCHS, weight_decay=0.01,
                    warmup_steps=WARMUP, fp16=True, eval_strategy='epoch', save_strategy='epoch',
                    logging_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='macro_f1',
                    greater_is_better=True, save_total_limit=2, seed=seed, report_to='none')
    settings.update(overrides)
    return WeightedTrainer(model=model, args=TrainingArguments(**settings), train_dataset=hf['train'],
                           eval_dataset=hf['val'], data_collator=collator, processing_class=tok,
                           compute_metrics=compute_metrics)

534 steps per epoch, 2136 in total, 213 warm-up steps, learning rate 2e-05


In [6]:
smoke = make_trainer(0, '/content/smoke', max_steps=40, warmup_steps=4, eval_strategy='no',
                     save_strategy='no', logging_strategy='steps', logging_steps=10, load_best_model_at_end=False)
start = time.perf_counter()
smoke.train()
sec_per_step = (time.perf_counter() - start) / 40
print(f'\n{sec_per_step:.2f} s per step -> about {sec_per_step * STEPS_PER_EPOCH * EPOCHS / 60:.0f} min of training per seed')
print('Validation after 40 steps:', {k: round(v, 4) for k, v in smoke.evaluate().items() if k in ('eval_macro_f1', 'eval_loss')})
print(f'Free space reported for Drive: {shutil.disk_usage(ROOT).free / 1e9:.1f} GB')
del smoke; torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
10,1.102116
20,1.091447
30,1.100028
40,1.092211



0.27 s per step -> about 9 min of training per seed


Training Loss,Validation Loss,Step,Macro F1,Accuracy
1.092211,1.095877,40,0.186614,0.388738


Validation after 40 steps: {'eval_loss': 1.0959, 'eval_macro_f1': 0.1866}
Free space reported for Drive: 5.1 GB


In [7]:
from datetime import date

CKPT_DIR = '/content/checkpoints'          # Colab's local disk, not Drive
os.makedirs(PRED, exist_ok=True)
log = f'{LOGS}/experiments.csv'

def log_run(row):
    logged = pd.read_csv(log)
    logged = logged[logged['run_id'].astype(str) != row['run_id']]
    pd.concat([logged, pd.DataFrame([row])]).to_csv(log, index=False)

def prediction_frame(split, proba, seed, model_label):
    df = splits[split]
    return pd.DataFrame({
        'id': df['id'], 'true_label': df['label'], 'pred_label': proba.argmax(axis=1),
        'p_neg': proba[:, 0].round(6), 'p_neu': proba[:, 1].round(6), 'p_pos': proba[:, 2].round(6),
        'model': model_label, 'seed': seed, 'train_data': 'sst5', 'eval_set': f'sst5_{split}',
    })

def run_seeds(lr, suffix=''):
    results = []
    for seed in [42, 43, 44]:
        tag = f'{MODEL_NAME}{suffix}_seed{seed}'
        out_dir, final_dir = f'{CKPT_DIR}/{tag}', f'{MODELS}/{tag}'
        test_file, hist_file = f'{PRED}/{tag}_sst5_test.csv', f'{LOGS}/{tag}_epochs.json'

        if os.path.exists(test_file) and os.path.exists(hist_file):
            print(f'{tag} already finished, loading its saved results')
            with open(hist_file) as f:
                results.append(json.load(f)['summary'])
            continue

        trainer = make_trainer(seed, out_dir, lr=lr)
        start = time.perf_counter()
        trainer.train()
        minutes = (time.perf_counter() - start) / 60

        evals = [h for h in trainer.state.log_history if 'eval_macro_f1' in h]
        train_losses = [h['loss'] for h in trainer.state.log_history if 'loss' in h]
        for h, tl in zip(evals, train_losses):
            print(f"  {tag} epoch {h['epoch']:.0f}: train loss {tl:.4f}, val loss {h['eval_loss']:.4f}, val macro-F1 {h['eval_macro_f1']:.4f}")
        best_epoch = int(max(evals, key=lambda h: h['eval_macro_f1'])['epoch'])

        trainer.save_model(final_dir)
        scores = {}
        for s in ['val', 'test']:
            logits = trainer.predict(hf[s]).predictions
            proba = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
            prediction_frame(s, proba, seed, MODEL_NAME + suffix).to_csv(f'{PRED}/{tag}_sst5_{s}.csv', index=False)
            scores[s] = f1_score(splits[s]['label'], proba.argmax(axis=1), average='macro')

        log_run({'run_id': f'{MODEL_NAME}{suffix}-sst5-seed{seed}', 'model': MODEL_NAME + suffix, 'seed': seed,
                 'train_data': 'sst5',
                 'hyperparameters': f'{CKPT}; adamw {lr}; batch 16; wd 0.01; warmup 10%; fp16; weighted ce; '
                                    f'max len {MAX_LEN}; 4 epochs, best by val macro-F1 (epoch {best_epoch})',
                 'val_macro_f1': round(scores['val'], 4), 'test_macro_f1': round(scores['test'], 4),
                 'date': date.today().isoformat()})
        summary = {'seed': seed, 'lr': lr, 'best_epoch': best_epoch,
                   'epoch1_train_loss': round(train_losses[0], 4),
                   'max_val_macro_f1': round(max(h['eval_macro_f1'] for h in evals), 4),
                   'val_macro_f1': round(scores['val'], 4), 'test_macro_f1': round(scores['test'], 4),
                   'minutes': round(minutes, 1)}
        with open(hist_file, 'w') as f:
            json.dump({'summary': summary, 'epochs': evals, 'train_losses': train_losses}, f, indent=2)
        shutil.rmtree(out_dir, ignore_errors=True)
        results.append(summary)
        print(summary, '\n')
        del trainer; torch.cuda.empty_cache()

    res = pd.DataFrame(results)
    res['collapsed'] = (res['max_val_macro_f1'] <= 0.40) | (res['epoch1_train_loss'] > 1.00)
    print(res.to_string(index=False))
    print(f"\nTest macro-F1: mean {res['test_macro_f1'].mean():.4f}, std {res['test_macro_f1'].std():.4f}")
    return res

res = run_seeds(LR)
if res['collapsed'].any():
    print('\nStability rule TRIGGERED: retrain all three seeds at 1e-5 (next step). The 2e-5 results stay on record.')
else:
    print('\nNo seed collapsed. Under the stability rule, the 2e-5 results stand.')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.872053,0.744108,0.666489,0.728429
2,0.629809,0.913020,0.666548,0.750227
3,0.469363,0.973062,0.680972,0.745686
4,0.348070,1.070918,0.686677,0.742053


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (20 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  roberta_seed44 epoch 1: train loss 0.8711, val loss 0.8412, val macro-F1 0.6542
  roberta_seed44 epoch 2: train loss 0.6320, val loss 0.7240, val macro-F1 0.6857
  roberta_seed44 epoch 3: train loss 0.4575, val loss 0.9512, val macro-F1 0.6807
  roberta_seed44 epoch 4: train loss 0.3397, val loss 1.0723, val macro-F1 0.6734


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'seed': 44, 'lr': 2e-05, 'best_epoch': 2, 'epoch1_train_loss': 0.8711, 'max_val_macro_f1': 0.6857, 'val_macro_f1': 0.6857, 'test_macro_f1': 0.7113, 'minutes': 4.6} 

 seed      lr  best_epoch  epoch1_train_loss  max_val_macro_f1  val_macro_f1  test_macro_f1  minutes  collapsed
   42 0.00002           4             0.8721            0.6867        0.6867         0.7209      4.8      False
   43 0.00002           3             0.8725            0.6791        0.6791         0.7052      4.2      False
   44 0.00002           2             0.8711            0.6857        0.6857         0.7113      4.6      False

Test macro-F1: mean 0.7125, std 0.0079

No seed collapsed. Under the stability rule, the 2e-5 results stand.


In [8]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

runs = [('svm', '', f'{PRED}/svm_sst5_test.csv')] + \
       [(m, s, f'{PRED}/{m}_seed{s}_sst5_test.csv') for m in ['lstm', 'distilbert', 'roberta'] for s in [42, 43, 44]]

rows = []
for model_name, seed, path in runs:
    p = pd.read_csv(path)
    prec, rec, f1, _ = precision_recall_fscore_support(p['true_label'], p['pred_label'], labels=[0, 1, 2], zero_division=0)
    rows.append({'model': model_name, 'seed': seed, 'neg_f1': f1[0], 'neu_recall': rec[1],
                 'neu_precision': prec[1], 'neu_f1': f1[1], 'pos_f1': f1[2], 'macro_f1': f1.mean()})
table = pd.DataFrame(rows)

cols = ['neg_f1', 'neu_recall', 'neu_precision', 'neu_f1', 'pos_f1', 'macro_f1']
print(table[table['model'] == 'roberta'].round(4).to_string(index=False))
print('\nMean and std across seeds:')
print(table.groupby('model', sort=False)[cols].agg(['mean', 'std']).round(4).to_string())

r = table[table['model'] == 'roberta']
mid = int(r.iloc[(r['macro_f1'] - r['macro_f1'].mean()).abs().argmin()]['seed'])
p = pd.read_csv(f'{PRED}/roberta_seed{mid}_sst5_test.csv')
print(f'\nRoBERTa seed {mid}, test confusion matrix (rows = true, columns = predicted):')
print(pd.DataFrame(confusion_matrix(p['true_label'], p['pred_label']),
                   index=['true neg', 'true neu', 'true pos'], columns=['pred neg', 'pred neu', 'pred pos']))

  model seed  neg_f1  neu_recall  neu_precision  neu_f1  pos_f1  macro_f1
roberta   42  0.8145      0.4807         0.4231  0.4501  0.8983    0.7209
roberta   43  0.8131      0.4987         0.3804  0.4316  0.8711    0.7052
roberta   44  0.8030      0.5450         0.3876  0.4530  0.8778    0.7113

Mean and std across seeds:
            neg_f1         neu_recall         neu_precision          neu_f1          pos_f1         macro_f1        
              mean     std       mean     std          mean     std    mean     std    mean     std     mean     std
model                                                                                                               
svm         0.6920     NaN     0.1414     NaN        0.2880     NaN  0.1897     NaN  0.7382     NaN   0.5399     NaN
lstm        0.6679  0.0558     0.3642  0.1737        0.3124  0.0615  0.3080  0.0678  0.7376  0.0019   0.5712  0.0040
distilbert  0.7666  0.0189     0.4302  0.0317        0.3487  0.0174  0.3848  0.0192  0.8412